# Agent Sandbox 浏览器沙箱教程

## 概述

本教程将指导您如何使用腾讯云 Python SDK 启动和使用**浏览器沙箱**。浏览器沙箱是一个安全的浏览器执行环境，为 AI Agent 提供完整的浏览器自动化能力。

### 什么是浏览器沙箱？

- **类型**: `browser` - 内置浏览器服务的沙箱工具
- **用途**: 为 AI Agent 提供安全、隔离的浏览器运行环境
- **优势**: 支持 VNC 可视化操作和 CDP 协议程序化控制

### 主要特性

- **可视化界面**: 通过 VNC 提供完整的桌面环境，可直接操作浏览器
- **程序化控制**: 支持 CDP（Chrome DevTools Protocol），自动化浏览器操作
- **文件系统集成**: 与沙箱文件系统无缝集成，支持文件上传下载
- **安全隔离**: 在隔离环境中运行，保证安全性

### 使用场景

- 需要登录状态的网页自动化
- 数据抓取和爬虫
- 浏览器操作结果保存到沙箱文件系统
- 需要可视化调试的自动化任务

### 前置条件

在使用本教程之前，您需要先在 [Agent Sandbox 控制台](https://console.cloud.tencent.com/) 中创建一个 `browser` 类型的沙箱工具。

### 教程大纲

1. 环境准备（安装 SDK、配置凭证）
2. 沙箱实例管理（启动、查询、获取令牌）
3. 浏览器操作（VNC 可视化、CDP 程序化控制）
4. 文件上传与浏览器集成
5. 清理资源

## 1. 环境准备

### 1.1 安装和验证依赖

In [ ]:
%pip install 'tencentcloud-sdk-python>=3.1.32' playwright
# 验证 SDK 安装
try:
    import tencentcloud
    from tencentcloud.ags.v20250920 import ags_client, models
    print(f"✅ 腾讯云 SDK 版本: {tencentcloud.__version__}")
    print("✅ AGS 模块导入成功")
    print("✅ SDK 版本: v20250920")
except ImportError as e:
    print(f"❌ 导入失败: {e}")
    print("请确保已安装腾讯云 SDK: pip install 'tencentcloud-sdk-python>=3.1.32'")

try:
    import playwright
    print("✅ Playwright 导入成功")
except ImportError as e:
    print(f"❌ Playwright 导入失败: {e}")
    print("请安装 Playwright: pip install playwright")

### 1.2 配置访问凭证

在使用 Agent Sandbox API 之前，您需要获取访问凭证：

1. 登录 [腾讯云控制台](https://console.cloud.tencent.com/cam/capi)
2. 进入「访问管理」→「API 密钥管理」
3. 创建或查看您的 SecretId 和 SecretKey

In [ ]:
import os
from tencentcloud.common import credential
from tencentcloud.common.profile.client_profile import ClientProfile
from tencentcloud.common.profile.http_profile import HttpProfile
from tencentcloud.common.exception.tencent_cloud_sdk_exception import TencentCloudSDKException
from tencentcloud.ags.v20250920 import ags_client, models

# 配置访问凭证（推荐使用环境变量）
if not os.getenv("TENCENTCLOUD_SECRET_ID"):
    os.environ["TENCENTCLOUD_SECRET_ID"] = "your_secret_id"  # 请替换为您的 SecretId
if not os.getenv("TENCENTCLOUD_SECRET_KEY"):
    os.environ["TENCENTCLOUD_SECRET_KEY"] = "your_secret_key"  # 请替换为您的 SecretKey
if not os.getenv("TENCENTCLOUD_REGION"):
    os.environ["TENCENTCLOUD_REGION"] = "ap-guangzhou"

# 创建认证对象
cred = credential.Credential(
    os.environ.get("TENCENTCLOUD_SECRET_ID"),
    os.environ.get("TENCENTCLOUD_SECRET_KEY")
)

# 配置 HTTP 选项
http_profile = HttpProfile()
http_profile.endpoint = "ags.tencentcloudapi.com"  # Agent Sandbox 服务域名

# 配置客户端选项
client_profile = ClientProfile()
client_profile.httpProfile = http_profile

# 创建 AGS 客户端
client = ags_client.AgsClient(cred, os.getenv("TENCENTCLOUD_REGION"), client_profile)

print("✅ 访问凭证配置完成")
print(f"📍 服务端点: {http_profile.endpoint}")
print(f"🌍 地域: {os.getenv('TENCENTCLOUD_REGION')}")
print("✅ AGS 客户端创建成功")

### 1.3 配置浏览器沙箱参数

请将 `SANDBOX_TOOL_NAME` 设置为您在控制台中已创建的浏览器沙箱工具名称：

In [ ]:
# ========================================
# 浏览器沙箱配置参数（统一管理）
# ========================================

# 请替换为您在控制台中已创建的浏览器沙箱工具名称
os.environ["SANDBOX_TOOL_NAME"] = "browser-v1"

# 实例配置
os.environ["SANDBOX_INSTANCE_TIMEOUT"] = "1h"  # 实例超时时间

print("✅ 浏览器沙箱配置参数设置完成")
print(f"🔧 工具名称: {os.getenv('SANDBOX_TOOL_NAME')}")
print(f"⏱️  实例超时: {os.getenv('SANDBOX_INSTANCE_TIMEOUT')}")

## 2. 沙箱实例管理

### 2.1 启动浏览器沙箱实例

使用已创建的浏览器工具启动沙箱实例。启动实例需要：
- `ToolId` 或 `ToolName`：指定要使用的沙箱工具（二选一）
- `Timeout`：实例超时时间

In [ ]:
import json

def start_sandbox_instance(tool_name=None, tool_id=None, timeout=None):
    """
    启动沙箱实例（使用环境变量配置）
    
    Args:
        tool_name (str): 工具名称（与 tool_id 二选一，默认使用环境变量）
        tool_id (str): 工具ID（与 tool_name 二选一）
        timeout (str): 超时时间（默认使用环境变量）
    
    Returns:
        response: 启动结果
    """
    try:
        # 使用环境变量作为默认值
        if not tool_name and not tool_id:
            tool_name = os.getenv("SANDBOX_TOOL_NAME")
        if not timeout:
            timeout = os.getenv("SANDBOX_INSTANCE_TIMEOUT")
        
        if not tool_name and not tool_id:
            raise ValueError("tool_name 和 tool_id 至少需要提供一个")
        
        # 创建请求对象
        req = models.StartSandboxInstanceRequest()
        
        # 设置工具标识
        if tool_id:
            req.ToolId = tool_id
        if tool_name:
            req.ToolName = tool_name
        
        # 设置超时时间
        req.Timeout = timeout
        
        # 发送启动请求
        resp = client.StartSandboxInstance(req)
        
        print(f"✅ 浏览器沙箱实例启动成功")
        if tool_name:
            print(f"🔧 使用工具: {tool_name}")
        if tool_id:
            print(f"🆔 工具ID: {tool_id}")
        print(f"⏱️  超时时间: {timeout}")
        print(f"📋 响应: {resp.to_json_string()}")
        
        return resp
        
    except TencentCloudSDKException as err:
        print(f"❌ 启动失败: {err}")
        return None
    except Exception as e:
        print(f"❌ 未知错误: {e}")
        return None

# 启动浏览器沙箱实例
print("=== 启动浏览器沙箱实例 ===")
instance_result = start_sandbox_instance()
instance_id = instance_result.Instance.InstanceId
print(f"\n🆔 实例ID: {instance_id}")

### 2.2 获取沙箱实例访问令牌

访问沙箱实例（VNC、CDP 等）需要先获取访问令牌（Access Token）。

In [ ]:
def acquire_sandbox_instance_token(instance_id):
    """
    获取沙箱实例访问令牌
    
    Args:
        instance_id (str): 实例ID
    
    Returns:
        response: 令牌信息
    """
    try:
        if not instance_id:
            raise ValueError("instance_id 是必需的")
        
        # 创建请求对象
        req = models.AcquireSandboxInstanceTokenRequest()
        req.InstanceId = instance_id
        
        # 发送请求
        resp = client.AcquireSandboxInstanceToken(req)
        
        print(f"✅ 沙箱实例令牌获取成功")
        print(f"🆔 实例ID: {instance_id}")
        print(f"📋 响应: {resp.to_json_string()}")
        
        return resp
        
    except TencentCloudSDKException as err:
        print(f"❌ 获取令牌失败: {err}")
        return None
    except Exception as e:
        print(f"❌ 未知错误: {e}")
        return None

# 获取访问令牌
print("=== 获取沙箱实例访问令牌 ===")
token_result = acquire_sandbox_instance_token(instance_id=instance_id)
sandbox_access_token = token_result.Token
print(f"🔑 令牌获取成功")

## 3. 浏览器操作

浏览器沙箱内置了浏览器服务，默认监听端口 `9000`，支持以下功能：
- **VNC 可视化界面**: 通过 noVNC 直接在浏览器中操作沙箱桌面
- **CDP 程序化控制**: 通过 Chrome DevTools Protocol 自动化浏览器操作

### 数据面访问域名格式

```
https://{port}-{instance_id}.{region}.tencentags.com
```

所有请求需携带 `X-Access-Token` 头部进行认证。

### 3.1 VNC 可视化界面

通过 VNC 可以直接操作沙箱内的浏览器，后续浏览器行为也可以通过 VNC 进行观察。

In [ ]:
from IPython.display import IFrame

region = os.getenv("TENCENTCLOUD_REGION")
browser_port = 9000  # 浏览器沙箱内置服务端口

# 构建 VNC 访问 URL
novnc_url = f"https://{browser_port}-{instance_id}.{region}.tencentags.com/novnc/vnc_lite.html?&path=websockify?access_token={sandbox_access_token}"
print(f"🖥️  VNC 访问地址: {novnc_url}")
print("ℹ️  您可以在浏览器中打开上述地址，直接操作沙箱桌面")

# 在 Jupyter Notebook 中嵌入 VNC 界面
IFrame(src=novnc_url, width="1920", height="1080")

### 3.2 基础浏览器控制（CDP）

通过 Chrome DevTools Protocol (CDP) 连接到沙箱浏览器，实现程序化控制。使用 Playwright 库进行浏览器自动化操作。

In [ ]:
from playwright.async_api import async_playwright

region = os.getenv("TENCENTCLOUD_REGION")
browser_port = 9000

# 构建 CDP 连接 URL
cdp_url = f"https://{browser_port}-{instance_id}.{region}.tencentags.com/cdp"
print(f"🔗 CDP 连接地址: {cdp_url}")

async with async_playwright() as playwright:
    # 通过 CDP 协议连接到远程浏览器
    browser = await playwright.chromium.connect_over_cdp(
        cdp_url,
        headers={"X-Access-Token": sandbox_access_token}
    )
    context = browser.contexts[0]
    page = context.pages[0]
    
    # 导航到指定页面
    await page.goto("https://tencent.com")
    await page.wait_for_load_state("networkidle")
    
    title = await page.title()
    print(f"✅ 页面加载成功")
    print(f"📄 页面标题: {title}")

### 3.3 文件上传与浏览器集成

将本地 HTML 文件上传到沙箱文件系统，然后在沙箱浏览器中打开。

上传文件到沙箱使用 envd API（端口 `49983`），通过 HTTP POST multipart/form-data 请求将文件写入指定路径。

In [ ]:
import urllib.request
import urllib.parse
import urllib.error
import uuid

# 1. 准备 HTML 文件内容
html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>Agent Sandbox Demo</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            text-align: center;
            padding: 50px;
            background-color: #f0f8ff;
        }
        h1 {
            color: #333;
            font-size: 3em;
        }
        p {
            color: #666;
            font-size: 1.2em;
        }
    </style>
</head>
<body>
    <h1>Welcome to Agent Sandbox</h1>
    <p>此页面通过云API SDK 上传到浏览器沙箱并在沙箱浏览器中打开</p>
</body>
</html>
"""

region = os.getenv("TENCENTCLOUD_REGION")
envd_port = 49983  # envd API 端口（文件操作）
browser_port = 9000  # 浏览器服务端口（CDP）

# 2. 上传文件到沙箱文件系统（通过 envd API，POST multipart/form-data）
file_path = "/home/user/demo.html"  # 沙箱内的文件路径
upload_url = f"https://{envd_port}-{instance_id}.{region}.tencentags.com/files?path={urllib.parse.quote(file_path)}&username=user"

try:
    # 构建 multipart/form-data 请求体
    boundary = uuid.uuid4().hex
    file_name = file_path.split('/')[-1]
    body = (
        f"--{boundary}\r\n"
        f"Content-Disposition: form-data; name=\"file\"; filename=\"{file_name}\"\r\n"
        f"Content-Type: application/octet-stream\r\n"
        f"\r\n"
        f"{html_content}\r\n"
        f"--{boundary}--\r\n"
    ).encode("utf-8")
    
    upload_req = urllib.request.Request(
        url=upload_url,
        data=body,
        headers={
            "X-Access-Token": sandbox_access_token,
            "Content-Type": f"multipart/form-data; boundary={boundary}"
        },
        method="POST"
    )
    with urllib.request.urlopen(upload_req) as response:
        print(f"✅ 文件上传成功，状态码: {response.getcode()}")
        print(f"📁 文件路径: {file_path}")
except Exception as e:
    print(f"❌ 文件上传失败: {e}")

# 3. 在沙箱浏览器中打开上传的 HTML 文件
cdp_url = f"https://{browser_port}-{instance_id}.{region}.tencentags.com/cdp"

async with async_playwright() as playwright:
    browser = await playwright.chromium.connect_over_cdp(
        cdp_url,
        headers={"X-Access-Token": sandbox_access_token}
    )
    context = browser.contexts[0]
    page = context.pages[0]
    
    # 打开沙箱内的 HTML 文件
    file_url = f"file://{file_path}"
    await page.goto(file_url)
    await page.wait_for_load_state("networkidle")
    
    title = await page.title()
    print(f"✅ 页面加载成功")
    print(f"📄 页面标题: {title}")

## 4. 清理资源

使用完毕后，及时停止沙箱实例释放资源。

In [ ]:
def stop_sandbox_instance(instance_id):
    """
    停止沙箱实例
    
    Args:
        instance_id (str): 实例ID
    
    Returns:
        response: 停止结果
    """
    try:
        if not instance_id:
            raise ValueError("instance_id 是必需的")
        
        # 创建请求对象
        req = models.StopSandboxInstanceRequest()
        req.InstanceId = instance_id
        
        # 发送停止请求
        resp = client.StopSandboxInstance(req)
        
        print(f"✅ 沙箱实例停止成功")
        print(f"🆔 已停止实例: {instance_id}")
        print(f"📋 响应: {resp.to_json_string()}")
        
        return resp
        
    except TencentCloudSDKException as err:
        print(f"❌ 停止失败: {err}")
        return None
    except Exception as e:
        print(f"❌ 未知错误: {e}")
        return None

# 停止浏览器沙箱实例
print("=== 停止浏览器沙箱实例 ===")
stop_result = stop_sandbox_instance(instance_id)